<html>
<head>
<meta charset="UTF-8">
<title>Software Village</title>
</head>

<body>

<div id="sv-container">
  <canvas id="sv-canvas" width="900" height="400"></canvas>

  <div id="sv-instructions">
    Use WASD to move. Press E near a cottage.
  </div>

  <div id="sv-question-panel">
    <button class="sv-btn" data-type="what">What is this?</button>
    <button class="sv-btn" data-type="download">How to download?</button>
    <button class="sv-btn" data-type="use">How to use?</button>
  </div>

  <div id="sv-answer-box"></div>
</div>

<script>
// Prevent space scroll
window.addEventListener("keydown", e => {
  if (e.code === "Space") e.preventDefault();
});

// Setup
const canvas = document.getElementById("sv-canvas");
const ctx = canvas.getContext("2d");

const instructions = document.getElementById("sv-instructions");
const questionPanel = document.getElementById("sv-question-panel");
const answerBox = document.getElementById("sv-answer-box");

const WIDTH = canvas.width;
const HEIGHT = canvas.height;

// Images
const bg = new Image();
bg.src = "/images/projects/gamify/software_images.png";

const avatar = new Image();
avatar.src = "/images/projects/gamify/man.png";

// Player
const player = {
  x: WIDTH / 2,
  y: HEIGHT / 2,
  speed: 2.2,
  frozen: false,
  size: 22
};

// Cottages
const cottages = [
  { id: "jupyter", x: WIDTH/2 - 80, y: 60 },
  { id: "git", x: 100, y: HEIGHT/2 - 40 },
  { id: "pip", x: WIDTH - 200, y: HEIGHT/2 - 40 },
  { id: "python", x: 100, y: HEIGHT - 160 },
  { id: "ruby", x: WIDTH - 200, y: HEIGHT - 160 }
];

const COTTAGE_ANSWERS = {
  jupyter: {
    what: "Jupyter Notebook is an interactive coding environment.",
    download: "pip install notebook or install Anaconda.",
    use: "Run: jupyter notebook"
  },
  git: {
    what: "Git tracks code changes.",
    download: "Download from git-scm.com",
    use: "git init, git add, git commit"
  },
  pip: {
    what: "Pip installs Python packages.",
    download: "Comes with Python",
    use: "pip install package_name"
  },
  python: {
    what: "Python is a programming language.",
    download: "Download from python.org",
    use: "python file.py"
  },
  ruby: {
    what: "Ruby is used for web apps.",
    download: "Download from ruby-lang.org",
    use: "ruby file.rb"
  }
};

// Input
const keys = {};
document.addEventListener("keydown", e => keys[e.key] = true);
document.addEventListener("keyup", e => keys[e.key] = false);

// Interaction
let interacting = false;
let currentCottage = null;

// Arrow
let arrowAngle = 0;
let arrowActive = false;
let arrowX = 0;
let arrowY = 0;
let arrowVX = 0;
let arrowVY = 0;

function fireArrow() {
  arrowActive = true;
  arrowX = player.x + Math.cos(arrowAngle) * 12;
  arrowY = player.y - 20 + Math.sin(arrowAngle) * 12;
  arrowVX = Math.cos(arrowAngle) * 6;
  arrowVY = Math.sin(arrowAngle) * 6;
}

// Key controls
document.addEventListener("keydown", e => {

  // SPACE to exit after answering
  if (e.code === "Space" && answerBox.style.display === "block") {
    endInteraction();
    return;
  }

  if (!interacting) return;

  // Rotate with A/D
  if (e.key === "a" || e.key === "A") arrowAngle -= 0.1;
  if (e.key === "d" || e.key === "D") arrowAngle += 0.1;

  // Shoot
  if (e.key === " ") fireArrow();

  if (e.key === "Escape") endInteraction();
});

// Buttons
document.querySelectorAll(".sv-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    showAnswer(btn.dataset.type);
  });
});

// Show answer
function showAnswer(type) {
  answerBox.textContent = COTTAGE_ANSWERS[currentCottage][type];
  answerBox.style.display = "block";
}

// End interaction
function endInteraction() {
  interacting = false;
  arrowActive = false;
  questionPanel.style.display = "none";
  answerBox.style.display = "none";
  instructions.textContent = "Use WASD to move. Press E near a cottage.";
  player.frozen = false;
}

// Detect cottage
function nearCottage() {
  for (let c of cottages) {
    if (Math.hypot(player.x - c.x, player.y - c.y) < 50) return c.id;
  }
  return null;
}

// Enter interaction
document.addEventListener("keydown", e => {
  if (e.key !== "e" && e.key !== "E") return;

  const c = nearCottage();
  if (!c) return;

  interacting = true;
  currentCottage = c;
  player.frozen = true;

  questionPanel.style.display = "flex";
  answerBox.style.display = "none";

  instructions.textContent = "Use A/D to aim, SPACE to shoot.";
});

// Arrow update
function updateArrow() {
  if (!arrowActive) return;

  arrowX += arrowVX;
  arrowY += arrowVY;

  const buttons = document.querySelectorAll(".sv-btn");
  const rectCanvas = canvas.getBoundingClientRect();

  buttons.forEach(btn => {
    const rect = btn.getBoundingClientRect();
    const bx1 = rect.left - rectCanvas.left;
    const by1 = rect.top - rectCanvas.top;
    const bx2 = bx1 + rect.width;
    const by2 = by1 + rect.height;

    if (arrowX > bx1 && arrowX < bx2 && arrowY > by1 && arrowY < by2) {
      arrowActive = false;
      showAnswer(btn.dataset.type);
    }
  });

  if (arrowX < 0 || arrowX > WIDTH || arrowY < 0 || arrowY > HEIGHT) {
    arrowActive = false;
  }
}

// Draw
function draw() {
  ctx.clearRect(0, 0, WIDTH, HEIGHT);

  ctx.drawImage(bg, 0, 0, WIDTH, HEIGHT);

  // Player
  ctx.beginPath();
  ctx.arc(player.x, player.y, player.size + 6, 0, Math.PI * 2);
  ctx.fillStyle = "rgba(0,0,0,0.6)";
  ctx.fill();

  ctx.drawImage(avatar, player.x - 20, player.y - 20, 40, 40);

  // Cottages (only when NOT interacting)
  if (!interacting) {
    ctx.strokeStyle = "black";
    ctx.lineWidth = 2;
    cottages.forEach(c => {
      ctx.strokeRect(c.x - 20, c.y - 20, 40, 40);
    });
  }

  // Arrow
  if (interacting) {
    ctx.save();
    ctx.translate(player.x, player.y - 20);
    ctx.rotate(arrowAngle);

    ctx.beginPath();
    ctx.moveTo(0, -12);
    ctx.lineTo(6, 6);
    ctx.lineTo(-6, 6);
    ctx.closePath();
    ctx.fillStyle = "white";
    ctx.fill();

    ctx.restore();
  }

  // Projectile
  if (arrowActive) {
    ctx.beginPath();
    ctx.arc(arrowX, arrowY, 4, 0, Math.PI * 2);
    ctx.fillStyle = "white";
    ctx.fill();
  }
}

// Loop
function loop() {
  if (!player.frozen) {
    if (keys["w"] || keys["W"]) player.y -= player.speed;
    if (keys["s"] || keys["S"]) player.y += player.speed;
    if (keys["a"] || keys["A"]) player.x -= player.speed;
    if (keys["d"] || keys["D"]) player.x += player.speed;
  }

  updateArrow();
  draw();
  requestAnimationFrame(loop);
}

loop();
</script>

</body>
</html>

## 🔧 Tool Setup Troubleshooting Guide

Use this page if something is not working.  
Each section is independent — jump directly to the area you are stuck. 

```mermaid
flowchart TD
    %% GitHub Sources
    subgraph GitHub_Pages[GitHub: Open-Coding-Society/pages]
        A[Repo: pages]:::repo
    end

    subgraph GitHub_Template[GitHub: Open-Coding-Society/student]
        T[Template Repo: student]:::repo
    end

    subgraph GitHub_Student[GitHub: your-github-id/student]
        B[Repo: student]:::repo
    end

    %% Local Computer
    subgraph Local[Local Computer]
        subgraph opencs_dir[opencs/ directory]
            C[pages/]:::local
            Ccmd[VSCode Prep<br/><br/>./scripts/venv.sh<br/>source venv/bin/activate<br/>code .]:::cmd
        end
        subgraph user_dir[your-github-id/ directory]
            D[student/]:::local
            Dcmd[VSCode Prep<br/><br/>./scripts/venv.sh<br/>source venv/bin/activate<br/>code .]:::cmd
        end
    end

    %% Arrows: cloning
    A -.->|clone/pull only| C
    B <--> |clone, pull & push| D

    %% Arrows: template relationship
    T -.->|template→created| B

    %% Arrows: commands
    C --> Ccmd
    D <--> Dcmd

```

---

## GitHub Commit / Config Recovery

Use these commands if git commit is failing

✅ **Expectation**  
You have a GitHub username + email

```bash
git config --list` # shows your GitHub username + email.  
```

❌ **If not personalized, run to match your credentials:**  

```bash
git config --global user.name "your-github-id"        # change to your GitHub ID
git config --global user.email "you@example.com"  # change to your Email
```

Or run script that does the same thing from your project directory

```bash
./scripts/activate.sh # help setup git config options
```

---

## Directory + Clone Recovery

✅ Expectation: Project directory is setup correctly

You can cd into your personal directory, and an ls shows your repo folder (ex: student).

- In VSCode open this file (tool_setup-trouble.ipynb), run the script with comment "Check for presence of repo and virtual environment".
- Or step through commands in terminal to aid in your comprehension.

### Navigate

```bash
cd ~/your-github-id # change your-github-id to your user directory name
```

❌ If cd fails, run:

```bash
mkdir ~/your-github-id
cd ~/your-github-id
```

### Check for repo folder

```bash
ls # lshould show "student"
```

❌ If missing, run:

```bash
git clone https://github.com/your-github-id/student.git # change to personal location of repo
```

### Move into repo folder

```bash
cd ~/your-github-id/student
ls # see listing of project files
```

---

In [ ]:
%%script bash

# Location of student repo, change to match your personal setup
repoLocation=~/your-github-id/student

# Check for presence of repo and virtual environment 
commands=(
    
  "ls --color=none -a ${repoLocation}" 
  "ls --color=none ${repoLocation}/venv/bin/activate" 
)

for cmd in "${commands[@]}"; do
  echo "### Command: $cmd"
  bash -c "$cmd"
done

## Virtual Environment Recovery

✅ Expectation
Your terminal prompt shows (venv) prefix.

### Run Vitual Environment

```bash
cd ~/your-github-id/student
source venv/bin/activate
```

❌ If it fails

```bash
./scripts/venv.sh
source venv/bin/activate
```

### VSCode Launch and Memories

✅ Satisfying the pre-requisites

- In project directory of your repo `pwd`
- Sourcing virtual environment `source venv/bin/activate`
- Ensure your terminal prompt shows the active virtual environment `(venv)`.

You are now ready to load VSCode and build a proper memory to open your project.

```bash
code .
```

✅ Verify VSCode launch

- Terminal and presence of `(venv)` prompt
- Open your Jokes IPYNB notebook and select the Python kernel with the `venv` prefix.

❌ If you fail verification

You may have opened your repo project without activating the proper `(venv)` environment.

Check the `Recent` listings. If there are entries that look incorrect or outdated (bad memories), remove them all.

- Shift-Cmd-P (Mac) or Shift-Ctl-P (Windows, Linux)
- type: Clear Recently Open -- select and confirm
- close VSCode
- Repeat VSCode Launch and Memories

---

## Version Checks

✅ Expectation: Version output for all tools 

In VSCode open this file (tool_setup-trouble.ipynb),  run the script with comment "Version and configuration checks".

- Output is expected for each `### Command`
- Version may be slightly different, but ask if you are not sure
* Java and IJava kernels are required for CSA only

❌ If anything fails

Best course of action is to run the specific OS specific activate scripts from `opencs/pages` project

WSL or Linux Users:  Ubuntu, Kali, or Mint

```bash
./scripts/activate_ubuntu.sh # ubuntu script works for all 3 variants, as they use same apt package manager
```

MacOS users

```bash
./scripts/activate_macos.sh # macos
```

Problems with git config

```bash
./scripts/activate.sh # help setup git config options
```

In [ ]:
%%script bash

# Version and configuration checks 
commands=(
  "python --version" 
  "pip --version" 
  "ruby -v" 
  "bundle -v" 
  "gem -v" 
  "jupyter --version" 
  "jupyter kernelspec list" 
  "git config --global user.name" 
  "git config --global user.email"
)

for cmd in "${commands[@]}"; do
  echo "### Command: $cmd"
  bash -c "$cmd"
done

## 🖥️ Machine Verification

Check that your machine has all required tools installed and configured correctly.
Start `host.py` before clicking the button below.

<button id="mv-run-btn" onclick="mvRun()">▶ Run Verification</button>

<div id="mv-error" style="display:none;">
  <strong>⚠ Could not connect to host.py</strong>
  <p>Could not reach <code>localhost:6741/api/host</code>. Start the local server first:</p>
  <ol>
    <li>Open a terminal and navigate to the <code>flask</code> project folder</li>
    <li>Run: <code>./scripts/venv.sh</code> then <code>source venv/bin/activate</code></li>
    <li>Run: <code>python host.py</code></li>
    <li>Return here and click <strong>Run Verification</strong> again</li>
  </ol>
</div>

<div id="mv-results" style="display:none;"></div>

<div id="mv-summary" style="display:none;">
  <span id="mv-count-pass"></span>
  &nbsp;&nbsp;
  <span id="mv-count-fail"></span>
</div>

<!-- ── Progress Graph ─────────────────────────────────────── -->
<div id="prog-section">
  <div>
    <h3>Installation Progress</h3>
    <div>
      <select id="prog-chart-type">
        <option value="line">Line graph</option>
        <option value="bar">Bar graph</option>
        <option value="heatmap">Heatmap</option>
      </select>
      <button id="prog-reset-btn">Reset history</button>
    </div>
  </div>
  <div>
    <label>Student name:</label>
    <input id="prog-name" type="text" placeholder="Enter your name" />
  </div>
  <div id="prog-empty">
    No runs yet — click Run Verification above.
  </div>
  <div style="display:none;" id="prog-chart-wrap">
    <canvas id="prog-chart" role="img" aria-label="Software installation progress over time"></canvas>
  </div>
  <div id="prog-heatmap" style="display:none;"></div>
  <table id="prog-table" style="display:none;">
    <thead>
      <tr>
        <th>Student</th>
        <th>Installed</th>
        <th>/ Total</th>
        <th>Timestamp</th>
      </tr>
    </thead>
    <tbody id="prog-log-body"></tbody>
  </table>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>

<script>
(function() {
  const STORE_KEY = 'mv_progress_v1';
  const TOTAL = 9;
  let log = JSON.parse(localStorage.getItem(STORE_KEY) || '[]');
  let chartInst = null;

function save() { localStorage.setItem(STORE_KEY, JSON.stringify(log)); }

  function fmtDate(iso) {
    var d = new Date(iso);
    return d.toLocaleDateString(undefined,{month:'short',day:'numeric'})
      + ' ' + d.toLocaleTimeString(undefined,{hour:'2-digit',minute:'2-digit'});
  }

  function renderAll() {
    var type = document.getElementById('prog-chart-type').value;
    var empty = document.getElementById('prog-empty');
    var wrap  = document.getElementById('prog-chart-wrap');
    var canvas= document.getElementById('prog-chart');
    var hm    = document.getElementById('prog-heatmap');
    var tbl   = document.getElementById('prog-table');
    if (log.length === 0) {
      empty.style.display='block'; wrap.style.display='none';
      hm.style.display='none'; tbl.style.display='none';
      if (chartInst) { chartInst.destroy(); chartInst = null; }
      return;
    }
    empty.style.display = 'none';
    tbl.style.display = 'table';
    if (type === 'heatmap') {
      wrap.style.display='none'; hm.style.display='block';
      if (chartInst) { chartInst.destroy(); chartInst = null; }
      renderHeatmap();
    } else {
      hm.style.display='none'; wrap.style.display='block';
      renderChart(type);
    }
    renderTable();
  }

  function renderChart(type) {
    if (chartInst) chartInst.destroy();
    chartInst = new Chart(document.getElementById('prog-chart'), {
      type: type === 'bar' ? 'bar' : 'line',
      data: {
        labels: log.map(function(e){ return fmtDate(e.ts); }),
        datasets: [{
          label: 'Installed',
          data: log.map(function(e){ return e.count; }),
          borderColor: '#3fb950',
          backgroundColor: type==='bar' ? 'rgba(63,185,80,0.6)' : 'rgba(63,185,80,0.15)',
          fill: type==='line', tension: 0.35,
          pointRadius: 5, pointBackgroundColor: '#3fb950', borderWidth: 2,
        }]
      },
      options: {
        responsive: true, maintainAspectRatio: false,
        plugins: { legend: { display: false } },
        scales: {
          y: { min:0, max:TOTAL, ticks:{ stepSize:1, color:'#7d8590' }, grid:{ color:'rgba(255,255,255,0.06)' } },
          x: { ticks:{ color:'#7d8590', maxRotation:45, autoSkip:false }, grid:{ color:'rgba(255,255,255,0.06)' } }
        }
      }
    });
  }

  function renderHeatmap() {
    var days = {};
    log.forEach(function(e) {
      var day = e.ts.slice(0,10);
      if (!days[day] || e.count > days[day]) days[day] = e.count;
    });
    var sorted = Object.keys(days).sort();
    var html = '<div>';
    sorted.forEach(function(day) {
      var v = days[day]; var alpha = 0.1 + (v/TOTAL)*0.85;
      html += '<div title="'+day+': '+v+'/'+TOTAL+'">'
        + ''+v+''
        + ''+day.slice(5)+'</div>';
    });
    html += '</div><p>Best run per day. Darker = more installed.</p>';
    document.getElementById('prog-heatmap').innerHTML = html;
  }

  function renderTable() {
    document.getElementById('prog-log-body').innerHTML = log.slice().reverse().map(function(e) {
      return '<tr>'
        + '<td>'+e.student+'</td>'
        + '<td>'+e.count+'</td>'
        + '<td>'+e.total+'</td>'
        + '<td>'+fmtDate(e.ts)+'</td>'
        + '</tr>';
    }).join('');
  }

  document.getElementById('prog-chart-type').addEventListener('change', renderAll);
  document.getElementById('prog-reset-btn').addEventListener('click', function() {
    if (!confirm('Clear all saved progress history?')) return;
    log = []; save(); renderAll();
  });

  // Called by mvRun() after verification completes
  window.__mvRecordRun = function(passCount) {
    var name = document.getElementById('prog-name').value.trim() || 'Anonymous';
    var entry = { student: name, count: passCount, total: TOTAL, ts: new Date().toISOString() };
    log.push(entry);
    save();
    renderAll();
  };

  renderAll();
})();
</script>

<script>
async function mvRun() {
  var btn     = document.getElementById('mv-run-btn');
  var errBox  = document.getElementById('mv-error');
  var results = document.getElementById('mv-results');
  var summary = document.getElementById('mv-summary');

  errBox.style.display  = 'none';
  results.style.display = 'none';
  summary.style.display = 'none';
  btn.disabled = true;
  btn.textContent = 'Checking...';

  var data;
  try {
    var res = await fetch('http://localhost:6741/api/host', { signal: AbortSignal.timeout(8000) });
    if (!res.ok) throw new Error('bad response');
    data = await res.json();
  } catch(e) {
    errBox.style.display = 'block';
    btn.disabled = false;
    btn.textContent = '▶ Run Verification';
    return;
  }

  function card(label, toolObj) {
    if (!toolObj) return '';
    var installed = toolObj.installed;
    var value = toolObj.version || toolObj.value || '';
    var display = value ? (value.length > 38 ? value.slice(0,38)+'...' : value) : (installed ? 'installed' : 'not found');
    var color = installed ? '#3fb950' : '#f85149';
    var icon  = installed ? '✅' : '❌';
    return '<div>'
      + '<span>' + icon + '</span>'
      + '<div>'
      + '<div>' + label + '</div>'
      + '<div>' + display + '</div>'
      + '</div></div>';
  }

  var html = '<p>💻 '
    + (data.OS||'') + ' ' + (data['OS Release']||'') + ' · ' + (data.Architecture||'') + '</p>';

  html += '<p>Git</p>';
  html += card('git', data.git) + card('git user.name', data.git_user_name) + card('git user.email', data.git_user_email);

  html += '<p>Python & Pip</p>';
  html += card('Python', data.python) + card('pip', data.pip);

  html += '<p>Ruby & Bundler</p>';
  html += card('Ruby', data.ruby) + card('Bundler', data.bundler) + card('RubyGems', data.gem);

  html += '<p>Jupyter</p>';
  html += card('Jupyter CLI', data.jupyter);
  if (data.jupyter_kernelspecs && data.jupyter_kernelspecs.installed) {
    (data.jupyter_kernelspecs.value||'').split('\n').filter(function(l){return l.includes('/');}).forEach(function(line){
      var kname = line.trim().split(/\s+/)[0] || 'kernel';
      html += card('kernel: ' + kname, { installed: true, value: 'registered' });
    });
  }

  if (data.OS === 'Darwin') {
    html += '<p>macOS Tools</p>';
    html += card('Homebrew', data.brew) + card('Xcode CLI Tools', data.xcode_select);
  }

  results.innerHTML = html;
  results.style.display = 'block';

  var keys = ['git','git_user_name','git_user_email','python','pip','ruby','bundler','gem','jupyter'];
  if (data.OS === 'Darwin') keys.push('brew','xcode_select');
  var pass = 0, fail = 0;
  keys.forEach(function(k){ if(data[k]){ data[k].installed ? pass++ : fail++; } });

  document.getElementById('mv-count-pass').textContent = pass + ' passing';
  document.getElementById('mv-count-fail').textContent = fail > 0 ? fail + ' failing' : '';
  summary.style.display = 'block';
  
  // ✅ ADD THIS
  if (window.__mvRecordRun) {
    window.__mvRecordRun(pass);
  }
  
  btn.disabled = false;
  btn.textContent = '▶ Run Again';

}
</script>

<nav class="ocs__pager" aria-label="Lesson navigation">
  <a class="ocs__pager-link" href="{{site.baseurl}}/guides/vscode/">&larr; VS Code and Project Setup</a>
  <a class="ocs__pager-link next" href="{{site.baseurl}}/guides/postman/">Testing APIs with Postman &rarr;</a>
</nav>

[All Guides]({{site.baseurl}}/guides/)
